# Frequency-Wavenumber Spectra Diagnostic: LLC vs Emulators
Variables: KE = rho0/2 * (U^2 + V^2),  B = -g*sigma0/rho0

Outputs per variable:
1. Spectra grid: rows = depth k in [0,10,20,30,40,50], cols = LLC + emulators
2. Difference grid: rows = depth, cols = (LLC - emulator_n)
3. Error-vs-depth scatter: cols = emulators, x = mean/median diff, y = depth (0..50)

In [2]:
pip install fastjmd95

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 92.3 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ocean-emulators 1.0 requires numpy<2,>=1.26.4, but you have numpy 2.3.5 which is incompatible.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import xrft
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from dask.diagnostics import ProgressBar
from fastjmd95 import jmd95numba

In [2]:
# ============== LOAD LLC ==============
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
        'name': 'long_curriculum,ckpt_12',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=3,ckpt_12/predictions_4d.zarr',
        'desc': 'strides=3,long_curriculum_ckpt12'
    },
    {
        'name': 'long_curriculum,ckpt_22',
        'key': 'emulator_2',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=3,ckpt_22/predictions_4d.zarr',
        'desc': 'strides=3,long_curriculum_ckpt22'
    },
]

emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(int(t.year), int(t.month), int(t.day),
                     int(t.hour), int(t.minute), int(t.second))
        if hasattr(t, 'year') else pd.Timestamp(t).floor('s')
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)
common_times = llc_times_norm
for cfg in emulator_configs:
    et = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(et)
common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)
print(f"LLC subset to {len(common_times)} common times")

grid_vars = ['XC', 'YC', 'rA', 'Z']
emulator_patches = {}
for cfg in emulator_configs:
    pr = emulator_patches_raw[cfg['key']]
    pmask = normalize_times(pr.time.values).isin(common_times)
    p = pr.isel(time=pmask)
    for gv in grid_vars:
        p[gv] = llc_patch[gv]
    emulator_patches[cfg['key']] = p

emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded long_curriculum,ckpt_12: strides=3,long_curriculum_ckpt12
Loaded long_curriculum,ckpt_22: strides=3,long_curriculum_ckpt22
LLC subset to 16 common times

=== Setup complete: LLC + 2 emulators ===
  long_curriculum,ckpt_12 (emulator_1)
  long_curriculum,ckpt_22 (emulator_2)


In [3]:
selected_time_range = [0, 16]
stepping = 1
start_idx, end_idx = selected_time_range

llc_patch = llc_patch.isel(time=slice(start_idx, end_idx + 1, stepping))

emulator_patches_subset = {}
for key, patch in emulator_patches.items():
    safe_end_idx = min(end_idx, patch.sizes['time'] - 1)
    emulator_patches_subset[key] = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )
emulator_patches = emulator_patches_subset

min_time_len = min([llc_patch.sizes['time']] +
                   [p.sizes['time'] for p in emulator_patches.values()])
llc_patch = llc_patch.isel(time=slice(0, min_time_len))
emulator_patches = {k: p.isel(time=slice(0, min_time_len))
                    for k, p in emulator_patches.items()}

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"Final synchronized length = {min_time_len}")
print(f"LLC: {llc_patch.sizes['time']} times")
for name, key in emulator_info:
    print(f"  {name} ({key}): {emulator_patches[key].sizes['time']} times")

Final synchronized length = 16
LLC: 16 times
  long_curriculum,ckpt_12 (emulator_1): 16 times
  long_curriculum,ckpt_22 (emulator_2): 16 times


## Compute Kinetic Energy: KE = rho0/2 * (U^2 + V^2)

In [4]:
rho0 = 1025.0  # kg/m^3

for patch_name, patch in all_patches.items():
    print(f"Computing KE for {patch_name}...")
    U = patch['U'].values
    V = patch['V'].values
    KE = 0.5 * rho0 * (U**2 + V**2)
    patch['KE'] = (('time', 'k', 'j', 'i'), KE)
    print(f"  ✓ KE shape: {KE.shape}")
print("Done.")

Computing KE for llc...
  ✓ KE shape: (16, 51, 720, 720)
Computing KE for emulator_1...
  ✓ KE shape: (16, 51, 720, 720)
Computing KE for emulator_2...
  ✓ KE shape: (16, 51, 720, 720)
Done.


## Compute Buoyancy: B = -g * sigma0 / rho0

In [ ]:
G = 9.81

for patch_name, patch in all_patches.items():
    print(f"Computing Buoyancy for {patch_name}...")
    Salt = patch['Salt'].values
    Theta = patch['Theta'].values
    sigma0 = jmd95numba.rho(Salt, Theta, 0) - rho0
    B = -G * sigma0 / rho0
    patch['B'] = (('time', 'k', 'j', 'i'), B)
    print(f"  ✓ B shape: {B.shape}")
print("Done.")

Computing Buoyancy for llc...
  ✓ B shape: (16, 51, 720, 720)
Computing Buoyancy for emulator_1...


## Spectra computation helpers

In [ ]:
# Patch grid (LLC4320 face1 patch is ~720x720; we use uniform metric coords).
# We approximate dx from rA so that x1/y1 are in metres.

def build_metric_coords(patch):
    """Build approximate uniform x1/y1 coords in meters from rA mean."""
    dx = float(np.sqrt(np.nanmean(patch['rA'].values)))
    nj = patch.sizes['j']
    ni = patch.sizes['i']
    x1 = (np.arange(ni) - ni/2 + 0.5) * dx
    y1 = (np.arange(nj) - nj/2 + 0.5) * dx
    return x1, y1, dx


def azimuthal_avg(k, l, f, N, nfactor=4.0):
    """Azimuthally average 2-D spectrum slice f(l,k) -> 1-D in kr."""
    kk, ll = np.meshgrid(k, l)
    K = np.sqrt(kk**2 + ll**2)
    nbins = int(N / nfactor)
    if k.max() > l.max():
        ki = np.linspace(0., l.max(), nbins)
    else:
        ki = np.linspace(0., k.max(), nbins)
    kidx = np.digitize(K.ravel(), ki)
    area = np.bincount(kidx)
    kr = np.bincount(kidx, weights=K.ravel()) / np.maximum(area, 1)
    iso_f = np.ma.masked_invalid(
        np.bincount(kidx, weights=f.ravel()) / np.maximum(area, 1)
    ) * kr
    return kr, iso_f


def compute_iso_spectrum(da, x1, y1, time_s):
    """3-D power spectrum -> azimuthal average -> (omega, kr) in SI."""
    da = da.assign_coords(
        x1=('i', x1), y1=('j', y1), time=('time', time_s)
    ).swap_dims({'i': 'x1', 'j': 'y1'})
    da['x1'].attrs['units'] = 'm'
    da['y1'].attrs['units'] = 'm'
    da['time'].attrs['units'] = 's'
    da = da.chunk({'time': -1, 'y1': -1, 'x1': -1})

    with ProgressBar():
        ps3d = xrft.power_spectrum(
            da, dim=['x1', 'y1', 'time'],
            window=True, window_correction=True,
        ).compute()

    kx_vals = ps3d.freq_x1.values
    ky_vals = ps3d.freq_y1.values
    omega_vals = ps3d.freq_time.values
    nomega = len(omega_vals)
    nfactor = 4.0

    _kr, _ = azimuthal_avg(kx_vals, ky_vals,
                           ps3d.isel(freq_time=0).values,
                           len(kx_vals), nfactor)
    nkr = len(_kr)
    ps_iso = np.ma.zeros((nomega, nkr))
    for j in range(nomega):
        _, ps_iso[j, :] = azimuthal_avg(
            kx_vals, ky_vals,
            ps3d.isel(freq_time=j).values,
            len(kx_vals), nfactor,
        )
    _kr[0] = 0.0
    ps_iso_xr = xr.DataArray(
        np.array(ps_iso),
        coords={'freq_time': omega_vals, 'kr': _kr},
        dims=['freq_time', 'kr'],
    )
    ps_iso_xr['freq_time'].attrs['units'] = 'cycles/s'
    ps_iso_xr['kr'].attrs['units'] = 'cycles/m'
    return ps_iso_xr

## Compute spectra for KE and B at all depths for all patches

In [ ]:
SPECTRA_VARS = ['KE', 'B']
K_LEVELS_FIG = [0, 10, 20, 30, 40, 50]   # for spectra grids
K_LEVELS_ALL = list(range(0, 51))         # for error vs depth scatter

x1, y1, dx_m = build_metric_coords(llc_patch)
print(f"Approx dx = {dx_m:.1f} m")

time_vals = llc_patch.time.values
time_s = (time_vals - time_vals[0]) / np.timedelta64(1, 's') \
    if np.issubdtype(time_vals.dtype, np.datetime64) else \
    np.array([(pd.Timestamp(t.isoformat()) - pd.Timestamp(time_vals[0].isoformat())).total_seconds()
              for t in time_vals])

spectra_cache = {}  # spectra_cache[var][patch_key][k] = ps_iso_xr

for var in SPECTRA_VARS:
    spectra_cache[var] = {}
    for patch_name, patch in all_patches.items():
        spectra_cache[var][patch_name] = {}
        for k in K_LEVELS_ALL:
            print(f"  spectra: {var} {patch_name} k={k}")
            da = patch[var].isel(k=k)
            spectra_cache[var][patch_name][k] = compute_iso_spectrum(da, x1, y1, time_s)

print("All spectra computed.")

## Plot helpers

In [ ]:
SPEC_XLIM = [0.005, 0.25]   # cycles/km
SPEC_YLIM = [0.018, 0.5]    # cph

VAR_LABELS = {
    'KE': r'$$KE\ [\mathrm{J/m^3}]$$',
    'B':  r'$$b\ [\mathrm{m/s^2}]$$',
}


def get_plot_arrays(ps):
    """Return kr_km(+), omega_cph(+), Z_vp = |w_cph|*P/1e3 in positive quadrant."""
    kr = ps.kr.values * 1e3
    om = ps.freq_time.values * 3600.0
    iso_km = ps.values / 1e3
    pos_o = om > 0
    pos_k = kr > 0
    Z = iso_km[np.ix_(pos_o, pos_k)]
    krp = kr[pos_k]
    omp = om[pos_o]
    Z_vp = np.abs(omp)[:, None] * Z
    return krp, omp, Z_vp

## Figure 1 (per variable): spectra grid — rows = depth, cols = LLC + emulators

In [ ]:
os.makedirs('figs/fwn_spectra', exist_ok=True)

panel_keys = ['llc'] + [key for _, key in emulator_info]
panel_labels = ['LLC'] + [name for name, _ in emulator_info]

for var in SPECTRA_VARS:
    print(f"Plotting spectra grid for {var}...")
    nrows = len(K_LEVELS_FIG)
    ncols = len(panel_keys)

    # Global vmin/vmax across all panels
    all_vals = []
    for k in K_LEVELS_FIG:
        for pkey in panel_keys:
            ps = spectra_cache[var][pkey][k]
            _, _, Z_vp = get_plot_arrays(ps)
            v = Z_vp[np.isfinite(Z_vp) & (Z_vp > 0)]
            if v.size > 0:
                all_vals.append(v)
    if len(all_vals) == 0:
        print('  no data; skipping')
        continue
    all_vals = np.concatenate(all_vals)
    vmin = np.percentile(all_vals, 2)
    vmax = np.percentile(all_vals, 98)
    levels = np.power(10., np.linspace(np.log10(vmin), np.log10(vmax), 16))[:15]

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows), dpi=150)
    if nrows == 1: axes = axes.reshape(1, -1)
    if ncols == 1: axes = axes.reshape(-1, 1)

    for r, k in enumerate(K_LEVELS_FIG):
        for c, (pkey, plabel) in enumerate(zip(panel_keys, panel_labels)):
            ax = axes[r, c]
            ps = spectra_cache[var][pkey][k]
            krp, omp, Z_vp = get_plot_arrays(ps)
            cs = ax.contourf(
                krp, omp, Z_vp, levels=levels,
                norm=LogNorm(vmin=vmin, vmax=vmax),
                cmap='magma', extend='both',
            )
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlim(SPEC_XLIM); ax.set_ylim(SPEC_YLIM)
            ax.set_title(f'{plabel}  k={k}', fontsize=9)
            if r == nrows - 1:
                ax.set_xlabel('kr (cycles/km)', fontsize=8)
            if c == 0:
                ax.set_ylabel('omega (cph)', fontsize=8)
            ax.tick_params(labelsize=7)

    cbar = fig.colorbar(cs, ax=axes.ravel().tolist(),
                        orientation='vertical', fraction=0.015, pad=0.02)
    cbar.set_label(f'|omega|*kr*P  ({VAR_LABELS[var]})$^2$', fontsize=9)
    fig.suptitle(f'Frequency-Wavenumber Spectra — {var}', fontsize=12, y=0.995)
    out = f'figs/fwn_spectra/spectra_grid_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

## Figure 2 (per variable): difference grid — LLC - emulator_n, rows = depth

In [ ]:
for var in SPECTRA_VARS:
    print(f'Plotting difference grid for {var}...')
    nrows = len(K_LEVELS_FIG)
    ncols = n_emulators

    # Compute all diffs to set symmetric vmax
    diff_vals = []
    diffs_cache = {}
    for k in K_LEVELS_FIG:
        diffs_cache[k] = {}
        krp, omp, Z_llc = get_plot_arrays(spectra_cache[var]['llc'][k])
        for _, emu_key in emulator_info:
            _, _, Z_emu = get_plot_arrays(spectra_cache[var][emu_key][k])
            d = Z_llc - Z_emu
            diffs_cache[k][emu_key] = (krp, omp, d)
            v = d[np.isfinite(d)]
            if v.size > 0:
                diff_vals.append(np.abs(v))
    if len(diff_vals) == 0:
        print('  no data; skipping')
        continue
    abs_max = np.percentile(np.concatenate(diff_vals), 98)
    levels = np.linspace(-abs_max, abs_max, 21)

    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows), dpi=150)
    if nrows == 1 and ncols > 1: axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1: axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1: axes = axes.reshape(1, 1)

    for r, k in enumerate(K_LEVELS_FIG):
        for c, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[r, c]
            krp, omp, d = diffs_cache[k][emu_key]
            cs = ax.contourf(krp, omp, d, levels=levels,
                             cmap='bwr', extend='both',
                             vmin=-abs_max, vmax=abs_max)
            ax.set_xscale('log'); ax.set_yscale('log')
            ax.set_xlim(SPEC_XLIM); ax.set_ylim(SPEC_YLIM)
            ax.set_title(f'LLC - {emu_name}  k={k}', fontsize=9)
            if r == nrows - 1:
                ax.set_xlabel('kr (cycles/km)', fontsize=8)
            if c == 0:
                ax.set_ylabel('omega (cph)', fontsize=8)
            ax.tick_params(labelsize=7)

    cbar = fig.colorbar(cs, ax=axes.ravel().tolist(),
                        orientation='vertical', fraction=0.015, pad=0.02)
    cbar.set_label(f'Difference: LLC - emulator  ({var})', fontsize=9)
    fig.suptitle(f'Frequency-Wavenumber Spectra Difference — {var}',
                 fontsize=12, y=0.995)
    out = f'figs/fwn_spectra/spectra_diff_grid_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

## Figure 3 (per variable): error vs depth scatter
Each emulator panel: x = mean / median |LLC_spectrum - emulator_spectrum|, y = depth k (0..50).

In [ ]:
for var in SPECTRA_VARS:
    print(f'Plotting error-vs-depth for {var}...')
    fig, axes = plt.subplots(1, n_emulators,
                             figsize=(5*n_emulators, 7), dpi=150)
    if n_emulators == 1:
        axes = [axes]

    # First pass — collect for shared xlim
    err_data = {}  # emu_key -> (means, medians)
    for emu_name, emu_key in emulator_info:
        means = np.zeros(len(K_LEVELS_ALL))
        medians = np.zeros(len(K_LEVELS_ALL))
        for idx, k in enumerate(K_LEVELS_ALL):
            _, _, Z_llc = get_plot_arrays(spectra_cache[var]['llc'][k])
            _, _, Z_emu = get_plot_arrays(spectra_cache[var][emu_key][k])
            d = np.abs(Z_llc - Z_emu)
            d = d[np.isfinite(d)]
            means[idx] = np.nanmean(d) if d.size else np.nan
            medians[idx] = np.nanmedian(d) if d.size else np.nan
        err_data[emu_key] = (means, medians)

    all_err = np.concatenate([np.concatenate(v) for v in err_data.values()])
    all_err = all_err[np.isfinite(all_err)]
    xmin = 0
    xmax = float(np.nanmax(all_err)) * 1.05 if all_err.size else 1.0

    depths = np.array(K_LEVELS_ALL)
    for col, (emu_name, emu_key) in enumerate(emulator_info):
        ax = axes[col]
        means, medians = err_data[emu_key]
        ax.scatter(means, depths, color='blue', s=30, alpha=0.7,
                   label='Mean', zorder=3)
        ax.plot(means, depths, color='blue', alpha=0.4, linewidth=1.5)
        ax.scatter(medians, depths, color='red', s=30, alpha=0.7,
                   label='Median', zorder=3)
        ax.plot(medians, depths, color='red', alpha=0.4, linewidth=1.5)

        ax.set_title(f'{emu_name}  {var}', fontsize=9)
        ax.set_xlabel('|LLC - emulator| spectrum diff', fontsize=8)
        ax.set_ylabel('Depth k', fontsize=8)
        ax.set_ylim(max(K_LEVELS_ALL), 0)
        ax.set_xlim(xmin, xmax)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.legend(fontsize=7, loc='lower right')

    fig.suptitle(f'Spectral error vs depth — {var}', fontsize=11, y=1.0)
    plt.tight_layout()
    out = f'figs/fwn_spectra/spectra_error_vs_depth_{var}.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  ✓ saved {out}')

print('All spectra diagnostic figures saved to figs/fwn_spectra/')